# 06 — PuLID Hook Diagnostic

Localizes the MVP's `preserved_fraction = 0.0` bug. Implements Review 001 §4's recommended check:

> Generate one image with PuLID embedding from seed A and another from seed B —
> identical prompt, identical noise. If the two outputs look near-identical,
> PuLID hooks aren't wired in.

**What we measure:**
1. **Image A vs Image B (different seeds, same prompt+noise):** if pixel MSE ≈ 0 and ArcFace cosine sim ≈ 1, PuLID is being silently bypassed — text-prompt-only generation
2. **Seed A vs Image A (PuLID's job):** the actual preservation cosine — should be ≥ 0.5 for PuLID to be working
3. **Seed A vs Seed A (validator self-test):** must be ≈ 1.0; if not, the *validator's* alignment is broken (not the generator)

Running this against just 2 generations + 4 ArcFace embeddings — should take <10 min on the warm cluster.

In [ ]:
%pip install --force-reinstall --no-deps mediapipe==0.10.20
# PuLID's eva_clip module imports ftfy + regex + torchsde — not in cap's deps,
# not in DBR ML's bundled set. Without these, `from pulid.pipeline_flux import
# PuLIDPipeline` raises ModuleNotFoundError, which our generator catches in a
# blanket `except ImportError` and silently falls back to IP-Adapter. That's
# the root cause of the MVP's preserved_fraction = 0.0.
%pip install ftfy regex torchsde
%pip install /Workspace/Users/alenj00@safeway.com/counterfactual-audit-pipeline

In [ ]:
dbutils.library.restartPython()

## Setup — Volume cache + HF token + PuLID source on sys.path

In [ ]:
import os, sys
from pathlib import Path

# CRITICAL: HF_HOME must NOT be on a GCS-FUSE Volume — HF Hub's .incomplete
# temp-file pattern uses open("ab") which FUSE rejects with "File too large".
HF_HOME_LOCAL = "/local_disk0/hf_cache"
HF_CACHE_VOLUME = "/Volumes/ds_work/alenj00/cap_cache/hf_cache"
PULID_SRC = "/Volumes/ds_work/alenj00/cap_cache/pulid_src"
FAIRFACE_DIR = Path("/Volumes/ds_work/alenj00/cap_cache/seeds/fairface")
OUT_DIR = Path("/local_disk0/cap_pulid_diag")
OUT_DIR.mkdir(parents=True, exist_ok=True)
Path(HF_HOME_LOCAL).mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = HF_HOME_LOCAL
os.environ["HUGGINGFACE_HUB_CACHE"] = HF_HOME_LOCAL
os.environ["TRANSFORMERS_CACHE"] = HF_HOME_LOCAL
# Reduce CUDA fragmentation — saves several hundred MB on the L4 (the 22 GB
# usable VRAM is tight with FP8 Flux + ControlNet + T5).
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# optimum-quanto JIT-compiles CUDA extensions on first use. Default cache
# (~/.cache/torch_extensions) is on rootfs and fills up; /local_disk0 has only
# ~4 GB free after HF caches; the linker scratch needs more. /dev/shm is a 32 GB
# tmpfs (RAM-backed, but only consumes what we write — < 2 GB for the .so).
os.environ["TORCH_EXTENSIONS_DIR"] = "/dev/shm/torch_extensions"
os.environ["TMPDIR"] = "/dev/shm/tmp"
Path("/dev/shm/torch_extensions").mkdir(parents=True, exist_ok=True)
Path("/dev/shm/tmp").mkdir(parents=True, exist_ok=True)

HF_TOKEN = dbutils.secrets.get(scope="cap-secrets", key="hf-token")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

if PULID_SRC not in sys.path:
    sys.path.insert(0, PULID_SRC)

# Confirm pulid is importable BEFORE the generator's _load_pulid runs.
try:
    from pulid.pipeline_flux import PuLIDPipeline
    print(f"✓ pulid.pipeline_flux.PuLIDPipeline importable from {PULID_SRC}")
except Exception as e:
    print(f"✗ PuLID import FAILED: {e!r}")
    raise

# PuLID's PuLIDPipeline.__init__ + load_pretrain do hf_hub_download with
# RELATIVE local_dir='models'. CWD must be on /local_disk0 (non-FUSE).
PULID_WORKDIR = Path("/local_disk0/pulid_workdir")
PULID_WORKDIR.mkdir(parents=True, exist_ok=True)

seed_paths = sorted(FAIRFACE_DIR.glob("*.jpg"))
assert len(seed_paths) >= 2, f"Need ≥2 FairFace seeds at {FAIRFACE_DIR}"
SEED_A = seed_paths[0]
SEED_B = seed_paths[1]
print(f"HF_HOME (local):       {HF_HOME_LOCAL}")
print(f"Flux cache_dir:        {HF_CACHE_VOLUME}")
print(f"CUDA alloc conf:       {os.environ.get('PYTORCH_CUDA_ALLOC_CONF')}")
print(f"Seed A: {SEED_A.name}")
print(f"Seed B: {SEED_B.name}")

## Generate two images — same prompt, same noise, DIFFERENT seed identities

In [ ]:
import time
from cap.generator import FluxPuLIDNativeGenerator, GenerationRequest

# FluxPuLIDNativeGenerator (NEW) uses PuLID's native Flux loader + a vendored
# Flux subclass that applies BOTH pulid_ca id injection AND ControlNet block
# residuals at the right block boundaries. Replaces the broken
# FluxPuLIDControlNetGenerator path that silently fell back to text-only Flux
# because diffusers' FluxTransformer2DModel doesn't honor pulid_ca attributes.
gen = FluxPuLIDNativeGenerator(
    flux_model_name="flux-dev",
    controlnet_mode="pose",
    cache_dir=HF_CACHE_VOLUME,
    face_model_name="antelopev2",
    id_weight=1.0,
    controlnet_conditioning_scale=0.6,
)

# chdir before lazy-load so PuLID's relative `local_dir='models'` writes
# to /local_disk0/pulid_workdir/models/ (a non-FUSE filesystem).
saved_cwd = os.getcwd()
os.chdir(PULID_WORKDIR)
try:
    t0 = time.time()
    gen._lazy_load()
    print(f"Lazy load: {time.time() - t0:.1f}s")
finally:
    os.chdir(saved_cwd)

print(f"Model versions: {gen.model_versions()}")

# Identical prompt + noise + axis values; ONLY the seed image differs.
common_kwargs = dict(
    seed_prompt=None,
    counterfactual_axes={"skin_tone": [3]},
    fixed_attributes={
        "pose": "frontal",
        "expression": "neutral expression",
        "lighting": "soft even studio lighting",
    },
    seed=42,
    num_inference_steps=20,
    guidance_scale=3.5,
    width=768,
    height=768,  # diagnostic floor (CLAUDE.md item 4); 1024 won't fit FP8 Flux + BF16 ControlNet on 22 GB L4
)

req_a = GenerationRequest(seed_identity_id="diag_A", seed_image_path=str(SEED_A), **common_kwargs)
req_b = GenerationRequest(seed_identity_id="diag_B", seed_image_path=str(SEED_B), **common_kwargs)

os.chdir(PULID_WORKDIR)
try:
    t0 = time.time()
    result_a = gen.generate(req_a, OUT_DIR)
    print(f"Image A generated in {time.time() - t0:.1f}s → {result_a[0].image_path}")

    t0 = time.time()
    result_b = gen.generate(req_b, OUT_DIR)
    print(f"Image B generated in {time.time() - t0:.1f}s → {result_b[0].image_path}")
finally:
    os.chdir(saved_cwd)

img_a_path = result_a[0].image_path
img_b_path = result_b[0].image_path

## Measure: pixel similarity + ArcFace cosine sims

In [ ]:
import numpy as np
from PIL import Image

def to_arr(p):
    return np.asarray(Image.open(p).convert("RGB"), dtype=np.float32) / 255.0

arr_a = to_arr(img_a_path)
arr_b = to_arr(img_b_path)

# Pixel MSE: 0.0 = identical, 1.0 = maximum difference at full saturation.
# A sane Flux generation typically lands between 0.05 and 0.20 for two
# different prompts/seeds. < 0.01 means the two images are essentially equal.
mse = float(((arr_a - arr_b) ** 2).mean())
print(f"Pixel MSE (image A vs image B): {mse:.6f}")

# ArcFace cosine sims via the validator (uses the SAME antelopev2 we ran on
# the MVP, with the cache_dir staging fix).
from cap.validator import ArcFaceIdentityValidator
validator = ArcFaceIdentityValidator(
    model_name="antelopev2",
    threshold=0.5,
    root=HF_CACHE,
)
validator._lazy_load()

def emb(p):
    return validator.embed(p)

def cossim(a, b):
    if a is None or b is None:
        return float("nan")
    return float(np.dot(a, b))  # both are L2-normalized

e_seed_a = emb(SEED_A)
e_seed_b = emb(SEED_B)
e_img_a  = emb(img_a_path)
e_img_b  = emb(img_b_path)

self_sim = cossim(e_seed_a, e_seed_a)  # validator self-test (must be ~1.0)
ab_seeds = cossim(e_seed_a, e_seed_b)  # different ground-truth identities (~0)
img_to_img = cossim(e_img_a, e_img_b)  # ← the headline diagnostic
preserved_a = cossim(e_seed_a, e_img_a)  # what we want > 0.5
preserved_b = cossim(e_seed_b, e_img_b)

print()
print(f"ArcFace self-sim (seed A vs seed A):           {self_sim:.4f}  ← must be ~1.0")
print(f"ArcFace seed A vs seed B:                       {ab_seeds:.4f}  ← unrelated faces, ~0")
print(f"ArcFace seed A vs image A (PuLID's job):        {preserved_a:.4f}  ← want ≥ 0.5")
print(f"ArcFace seed B vs image B (PuLID's job):        {preserved_b:.4f}  ← want ≥ 0.5")
print(f"ArcFace image A vs image B (HEADLINE):          {img_to_img:.4f}")
print()
print("Diagnostic verdict:")
if mse < 0.01 or img_to_img > 0.95:
    print("  ✗ PuLID HOOKS NOT WIRED — same prompt + noise produced essentially the same face.")
    print("    The id_embedding is computed and passed but the PuLIDPipeline isn't actually")
    print("    modulating the Flux transformer. Inspect pulid.pipeline_flux's hook registration.")
elif self_sim < 0.95:
    print("  ✗ VALIDATOR ALIGNMENT BROKEN — seed-vs-self sim should be ~1.0 but isn't.")
    print("    Generator may be fine; the bug is in the ArcFace embedding / alignment path.")
elif preserved_a >= 0.5 and preserved_b >= 0.5:
    print("  ✓ PuLID IS WORKING — outputs differ AND preserve their respective seeds.")
    print("    The MVP's preserved_fraction=0 must be from a different bug (wrong-pair? scale?).")
else:
    print("  ⚠ AMBIGUOUS — outputs differ (PuLID is doing *something*) but preservation is")
    print("    below the threshold. Likely candidates: identity scale too low, alignment")
    print("    inconsistency between seed and generated face crops.")

## Visual side-by-side

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes[0, 0].imshow(Image.open(SEED_A)); axes[0, 0].set_title(f"Seed A: {SEED_A.name}"); axes[0, 0].axis('off')
axes[0, 1].imshow(Image.open(SEED_B)); axes[0, 1].set_title(f"Seed B: {SEED_B.name}"); axes[0, 1].axis('off')
axes[1, 0].imshow(Image.open(img_a_path)); axes[1, 0].set_title("Generated A (PuLID seed=A)"); axes[1, 0].axis('off')
axes[1, 1].imshow(Image.open(img_b_path)); axes[1, 1].set_title("Generated B (PuLID seed=B)"); axes[1, 1].axis('off')
plt.tight_layout()
plt.show()